# Data inspection: Scam Call Alert

One sample is one complete conversation. Input is Thai text; labels are 0 = Not Scam, 1 = Scam. Original categories are metadata only.

Read-only inspection: no raw changes, label corrections, exclusions or training exports. Generated review queues are pending human decisions. Keep executed outputs local.

Course basis: Chapter 2 pp. 29, 32-44, 46-54, 62-70; Lab 1 Tasks 1-2; Chapter 6 pp. 15-18. Run all cells from the repository root or dataset directory. Requires PyArrow and scikit-learn.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import csv, hashlib, json, re, platform
import importlib.metadata
import pyarrow.parquet as pq
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

ROOT = Path.cwd()
if ROOT.name == "02 - DATASETs":
    ROOT = ROOT.parent
DATA = ROOT / "02 - DATASETs"
manifest = json.loads((DATA / "sources.json").read_text())
hashes = {}
for source in manifest["datasets"]:
    for item in source["files"]:
        actual = hashlib.sha256((DATA / item["path"]).read_bytes()).hexdigest()
        assert actual == item["sha256"], f"Checksum mismatch: {item['path']}"
        hashes[item["path"]] = actual
primary = pq.read_table(DATA / "raw/scam-dialogue-thai-translated/data/train-00000-of-00001.parquet").to_pylist()
with (DATA / "raw/thai-call-center-call-log/Dataset.csv").open(encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)
    assert reader.fieldnames == ["Id", "Text", "Type"]
    supplemental = list(reader)
assert len(primary) == 1280 and len(supplemental) == 198
assert set(primary[0]) == {"dialogue", "type", "label", "translated dialogue", "translated type"}
mapping = {"Scam": 1, "General": 0, "Official": 0, "Verification-call": 0, "Unknow": None}
rows = []
for i, r in enumerate(primary):
    rows.append(dict(conversation_id=f"hf_{i:06d}", source="primary", source_record_id=str(i),
                     text=r["translated dialogue"], original_text=r["dialogue"],
                     original_label=r["label"], proposed_label=r["label"], original_category=r["type"]))
for i, r in enumerate(supplemental):
    assert r["Type"] in mapping, f"Unknown category: {r['Type']}"
    rows.append(dict(conversation_id=f"kaggle_{i:06d}", source="supplemental", source_record_id=r["Id"], source_row_index=i,
                     text=r["Text"], original_text=None, original_label=r["Type"],
                     proposed_label=mapping[r["Type"]], original_category=r["Type"]))
assert len({r["conversation_id"] for r in rows}) == len(rows), "Duplicate source IDs"
for source in ["primary", "supplemental"]:
    subset = [r for r in rows if r["source"] == source]
    print(source, len(subset), "original labels:", dict(Counter(str(r["original_label"]) for r in subset)))
    print("categories:", dict(Counter(r["original_category"] for r in subset)))


## Examples and integrity checks

Show at least four examples per source. Flags are review requests, not confirmed errors. Short text, English-only messages, missing speaker roles or a low translation length ratio may be legitimate.

Before/after demonstration: standardize line endings and trim surrounding whitespace only. Do not delete numbers, URLs, punctuation or scam signals.

In [ ]:
examples = []
for label in [0, 1]:
    examples += [r for r in rows if r["source"] == "primary" and r["proposed_label"] == label][:2]
for category in ["Scam", "General", "Official", "Unknow"]:
    examples.append(next(r for r in rows if r["source"] == "supplemental" and r["original_category"] == category))
for r in examples:
    print("\n", r["conversation_id"], "label:", r["original_label"], "category:", r["original_category"])
    print(r["text"])

def fixed_format(text):
    return text.replace("\r\n", "\n").replace("\r", "\n").strip()

def duplicate_key(text):
    return re.sub(r"\s+", " ", fixed_format(text))

assert fixed_format(" \r\nผู้โทร: OTP 123456 https://example.com\r\n ") == "ผู้โทร: OTP 123456 https://example.com"
flags = defaultdict(set)
source_id_counts = Counter(r["source_record_id"] for r in rows if r["source"] == "supplemental")
changed = []
for r in rows:
    cid, text = r["conversation_id"], r["text"]
    if r["source"] == "supplemental" and source_id_counts[r["source_record_id"]] > 1:
        flags[cid].add("duplicate_source_id")
    if not isinstance(text, str) or not text.strip():
        flags[cid].add("missing_or_invalid_text")
        continue
    if r["proposed_label"] not in (0, 1):
        flags[cid].add("unresolved_label")
    if r["source"] == "primary" and (type(r["original_label"]) is not int or r["original_label"] not in (0, 1)):
        flags[cid].add("invalid_binary_label")
    if not re.search(r"[\u0e00-\u0e7f]", text):
        flags[cid].add("no_thai_characters")
    if "\ufffd" in text or any(ord(c) < 32 and c not in "\n\r\t" for c in text):
        flags[cid].add("control_or_replacement_character")
    if len(text.strip()) < 100:
        flags[cid].add("short_text_under_100_characters")
    if r["source"] == "primary":
        if not re.search(r"(ผู้โทร|caller)\s*:", text, re.I) or not re.search(r"(ผู้รับ|receiver)\s*:", text, re.I):
            flags[cid].add("speaker_role_check")
        if not isinstance(r["original_text"], str) or not r["original_text"].strip():
            flags[cid].add("missing_original_text")
        elif len(text) / len(r["original_text"]) < 0.25:
            flags[cid].add("low_translation_length_ratio")
    elif not re.search(r"\[Call\]\(", text):
        flags[cid].add("speaker_role_check")
    if text != fixed_format(text):
        changed.append(cid)
print("Integrity flags:", dict(Counter(x for v in flags.values() for x in v)))
print("Minimal formatting affects rows:", len(changed))
for cid in changed[:4]:
    text = next(r["text"] for r in rows if r["conversation_id"] == cid)
    print(cid, "BEFORE", repr(text[:180]), "AFTER", repr(fixed_format(text)[:180]))


## Duplicate inspection

Exact matching collapses whitespace only, preserving case and content. Near candidates use character 3-5-gram TF-IDF cosine similarity >= 0.90 across both sources. This is a lexical screen, not a semantic duplicate guarantee. Human review must confirm groups; do not delete rows automatically.

The audit vectorizer is discarded and must never be reused as classifier preprocessing. Learned model transformations must be fitted within each training fold.

In [ ]:
groups = defaultdict(list)
valid = [r for r in rows if isinstance(r["text"], str) and r["text"].strip()]
for r in valid:
    groups[duplicate_key(r["text"])].append(r)
exact = []
for members in groups.values():
    if len(members) < 2:
        continue
    conflict = len({r["proposed_label"] for r in members if r["proposed_label"] is not None}) > 1
    exact.append(dict(candidate_group_id=f"exact_{len(exact):04d}", members=[r["conversation_id"] for r in members],
                      label_conflict=conflict, cross_source=len({r["source"] for r in members}) > 1))
    for r in members:
        flags[r["conversation_id"]].add("exact_duplicate")
        if conflict:
            flags[r["conversation_id"]].add("exact_duplicate_label_conflict")
texts = [duplicate_key(r["text"]) for r in valid]
features = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), lowercase=False).fit_transform(texts)
nn = NearestNeighbors(metric="cosine", algorithm="brute", radius=0.1000000001).fit(features)
distances, indices = nn.radius_neighbors(features)
near = []
for i, (ds, js) in enumerate(zip(distances, indices)):
    for d, j in zip(ds, js):
        j = int(j)
        if j <= i or texts[i] == texts[j] or 1 - d < 0.90:
            continue
        a, b = valid[i], valid[j]
        near.append(dict(left=a["conversation_id"], right=b["conversation_id"], similarity=round(float(1-d), 6),
                         label_conflict=a["proposed_label"] is not None and b["proposed_label"] is not None and a["proposed_label"] != b["proposed_label"],
                         cross_source=a["source"] != b["source"]))
        flags[a["conversation_id"]].add("near_duplicate_candidate")
        flags[b["conversation_id"]].add("near_duplicate_candidate")
near.sort(key=lambda p: (-p["similarity"], p["left"], p["right"]))
print("Exact groups:", len(exact), "redundant copies:", sum(len(g["members"])-1 for g in exact))
print("Exact label conflicts:", sum(g["label_conflict"] for g in exact))
print("Near pairs:", len(near), "label conflicts:", sum(p["label_conflict"] for p in near))
print("Top candidates:", near[:10])


## Review queue and report

Queue every supplemental row, all flagged primary rows, and five examples per primary category spread through source order. This initial review sample is not a statistical label-quality estimate. Decisions remain pending.

New review batch directories prevent reruns from overwriting manual decisions. Aggregate reports may be tracked; transcript-level review files and executed notebook remain local. No cleaned dataset, exclusions or final split groups are produced.

In [ ]:
sample_ids = set()
categories = defaultdict(list)
for r in rows:
    if r["source"] == "primary":
        categories[r["original_category"]].append(r)
for members in categories.values():
    for i in range(5):
        sample_ids.add(members[round(i*(len(members)-1)/4)]["conversation_id"])
queue = []
for r in rows:
    reasons = set(flags[r["conversation_id"]])
    if r["source"] == "supplemental":
        reasons.add("review_all_supplemental")
    if r["conversation_id"] in sample_ids:
        reasons.add("primary_category_sample")
    if reasons:
        queue.append({**r, "review_reasons": sorted(reasons), "decision": "pending",
                      "reviewed_label": None, "reviewer": "", "decision_reason": ""})
summary = dict(
    inspection_version="v1", status="automated_inspection_complete_manual_review_pending",
    source_checksums=hashes,
    software={"python": platform.python_version(), **{p: importlib.metadata.version(p) for p in ["pyarrow", "scikit-learn"]}},
    counts={s: sum(r["source"] == s for r in rows) for s in ["primary", "supplemental"]},
    source_label_counts={s: dict(Counter(str(r["original_label"]) for r in rows if r["source"] == s)) for s in ["primary", "supplemental"]},
    category_counts={s: dict(Counter(r["original_category"] for r in rows if r["source"] == s)) for s in ["primary", "supplemental"]},
    flag_counts=dict(sorted(Counter(x for v in flags.values() for x in v).items())),
    formatting_changed_rows=len(changed), exact_duplicate_groups=len(exact),
    exact_redundant_copies=sum(len(g["members"])-1 for g in exact),
    exact_label_conflict_groups=sum(g["label_conflict"] for g in exact),
    exact_cross_source_groups=sum(g["cross_source"] for g in exact),
    near_candidate_pairs=len(near), near_label_conflict_pairs=sum(p["label_conflict"] for p in near),
    near_cross_source_pairs=sum(p["cross_source"] for p in near),
    review_queue_counts=dict(Counter(r["source"] for r in queue)),
    rows_excluded=0, labels_changed=0, approved_training_rows=None)
reports = DATA / "reports"
reports.mkdir(exist_ok=True)
review_root = DATA / "review"
review_root.mkdir(exist_ok=True)
run = 1
while (review_root / f"inspection_{run:03d}").exists():
    run += 1
batch = review_root / f"inspection_{run:03d}"
batch.mkdir()
for name, items in [("review_queue.jsonl", queue), ("exact_duplicates.jsonl", exact), ("near_duplicate_candidates.jsonl", near)]:
    with (batch / name).open("w", encoding="utf-8") as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
(reports / "inspection_v1.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2)+"\n", encoding="utf-8")
lines = ["# Data quality inspection v1", "",
         "Historical initial inspection: pending flags below are a source snapshot, not current preparation decisions. See [the preparation findings](data_preparation_findings_v1.md) if prepared v1 exists.", "",
         "## Original and current counts", "",
         "| Source | Original rows | Excluded | Label changes |",
         "| --- | ---: | ---: | ---: |",
         f"| Primary | {len(primary)} | 0 | 0 |",
         f"| Supplemental | {len(supplemental)} | 0 | 0 |", "",
         "No rows have been changed or removed. Approved post-review counts remain undecided.", "",
         "## Automatic findings", "",
         f"- Exact duplicate groups: {len(exact)}; redundant copies: {summary['exact_redundant_copies']}.",
         f"- Exact conflicting-label groups: {summary['exact_label_conflict_groups']}.",
         f"- Near candidates: {len(near)} pairs; conflicting proposed labels: {summary['near_label_conflict_pairs']}.",
         f"- Cross-source matches: {summary['exact_cross_source_groups']} exact groups and {summary['near_cross_source_pairs']} near pairs.",
         f"- Minimal formatting would affect {len(changed)} rows; this was demonstrated only.",
         f"- Pending review queue: {len(queue)} rows, by source: {summary['review_queue_counts']}.", "",
         "| Review flag | Rows |", "| --- | ---: |"]
lines[-2:] = ["| Review flag | Rows | Source columns and affected conversations |", "| --- | ---: | --- |"]
review_link = f"../review/{batch.name}"
columns = {"duplicate_source_id": "Id, with Text and Type", "unresolved_label": "Type = Unknow, with Text"}
lines += [f"| {reason} | {count} | [{columns.get(reason, 'Text / translated dialogue')}]({review_link}/{reason}.md) |" for reason, count in summary["flag_counts"].items()]
lines += ["", "Links show full text, source row index, original ID, labels and flags. Primary entries also show the English original. These review pages stay local.",
          f"[Compare exact duplicate groups]({review_link}/exact_duplicates.md). [Compare near candidates]({review_link}/near_duplicate_candidates.md).",
          f"[Review all supplemental conversations]({review_link}/review_all_supplemental.md). [Review primary category samples]({review_link}/primary_category_sample.md).",
          f"Record decisions in [review_queue.jsonl]({review_link}/review_queue.jsonl) using the [labeling guideline](../labeling_guideline.md)."]
lines += ["", "Flags overlap and are not confirmed errors. Near candidates use character 3-5-gram TF-IDF cosine similarity >= 0.90, not semantic equivalence.", "",
          "## Five quality dimensions", "",
          "- Completeness: automated checks cover missing/invalid text and length flags. Semantic completeness and truncation require manual review.",
          "- Accuracy: source labels are retained and not independently validated. Supplemental binary labels remain proposals; Unknow is unresolved.",
          "- Consistency: schemas and unique generated IDs are checked; three Kaggle source IDs each occur twice. Original speaker tags remain; minimal formatting is demonstrated without a training export.",
          "- Representativeness: both datasets are simulated. Translated foreign scenarios, limited category coverage and unknown real-call prevalence constrain claims.",
          "- Reliability: source revisions, version and hashes are recorded. The translated card declares no license; Kaggle lists CC BY-NC-ND 4.0. Confirm rights before publishing derivatives.", "",
          "## Next steps", "",
          "Review the local queue using labeling_guideline.md. Record a reviewer, decision, reason and corrected label when needed. Inspect duplicate candidates in context before confirming groups.",
          "Then export versioned cleaned sources with exclusion/correction logs and before/after class and category counts. Keep related conversations and future chunks together during splitting.",
          "The audit vectorizer must not be used as classifier preprocessing. Fit learned transformations on training data within each fold.", "",
          "## Evidence", "",
          "Notebook: inspection.ipynb; local executed copy: inspection.executed.ipynb. Aggregate counts and software versions: inspection_v1.json. Local review batches: review/inspection_NNN/.",
          "Course basis: Chapter 2 pp. 29, 32-44, 46-54, 62-70; Lab 1 Tasks 1-2; Chapter 6 pp. 15-18.", ""]
(reports / "data_quality_v1.md").write_text("\n".join(lines), encoding="utf-8")
for relative, expected in hashes.items():
    assert hashlib.sha256((DATA / relative).read_bytes()).hexdigest() == expected
assert all(r["decision"] == "pending" for r in queue)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Local review batch:", batch.relative_to(ROOT))
print("Raw files unchanged; no split or cleaned training data created.")

# Readable local review pages. Escape HTML and table separators in source text.
from html import escape

def review_text(value):
    return escape(str(value)).replace("|", "&#124;").replace("\n", "<br>")

by_id = {r["conversation_id"]: r for r in queue}
def conversation_section(r):
    text_column = "translated dialogue" if r["source"] == "primary" else "Text"
    label_column = "label" if r["source"] == "primary" else "Type"
    index = r.get("source_row_index", int(r["source_record_id"]) if r["source"] == "primary" else None)
    lines = [
        f"### {r['conversation_id']}", "",
        f"- Source: {r['source']}; zero-based source row index: {index}.",
        f"- Original source ID: {r['source_record_id']}.",
        f"- Source label column: {label_column}; original value: {r['original_label']}.",
        f"- Proposed binary label: {r['proposed_label']}; decision: {r['decision']}.",
        f"- Original category: {r['original_category']}.",
        f"- Flags: {', '.join(r['review_reasons'])}.", "",
        f"**Source text column: {text_column}**", "",
        review_text(r["text"]), ""
    ]
    if r["original_text"] is not None:
        lines += ["**English original column: dialogue**", "", review_text(r["original_text"]), ""]
    return lines

for reason in sorted({reason for r in queue for reason in r["review_reasons"]}):
    members = [r for r in queue if reason in r["review_reasons"]]
    lines = [f"# Review: {reason}", "", f"{len(members)} conversations. Flags are not confirmed errors. No review decisions have been applied.",
             "", "Read-only view of the saved review queue. Record decisions in review_queue.jsonl using the labeling guideline.",
             "", "Source row indices are zero-based data-row positions, not physical CSV line numbers. Original Kaggle IDs may repeat.", ""]
    for r in members:
        lines += conversation_section(r)
    (batch / f"{reason}.md").write_text("\n".join(lines), encoding="utf-8")

for name, items in [("exact_duplicates", exact), ("near_duplicate_candidates", near)]:
    lines = [f"# Review: {name}", "", "Candidates require manual confirmation. Compare the full conversation texts below.", ""]
    for number, item in enumerate(items, 1):
        lines += [f"## Candidate {number}", "", "Metadata: " + review_text(json.dumps(item, ensure_ascii=False)), ""]
        ids = item["members"] if "members" in item else [item["left"], item["right"]]
        for cid in ids:
            lines += conversation_section(by_id[cid])
    (batch / f"{name}.md").write_text("\n".join(lines), encoding="utf-8")
